# CIMD interop walkthrough

**CIMD** is Client ID Metadata Documents,
`draft-ietf-oauth-client-id-metadata-document`, adopted by MCP as SEP-991. A
client publishes a small JSON document at an HTTPS URL, and that URL *is* its
`client_id`. No registration, and the authorization server stores nothing.

**Nothing in the authorization flow is hand-rolled.** The `mcp` SDK performs
discovery, chooses the `client_id`, builds the PKCE challenge and exchanges the
code. The discovery cells below make the same calls a second time so you can
read them; they observe, and are not what is under test.

---

## Start these two first

The document has to be reachable from the public internet, because the
authorization server dereferences the `client_id` URL itself from inside its
own cluster. `client_metadata.py` serves it and nothing else.

```bash
# terminal 1
./.venv/bin/python -m uvicorn client_metadata:app --port 8800
```
```bash
# terminal 2
ngrok http 8800
```

Copy the `https://` URL ngrok prints into `DOCUMENT_URL` below, with
`/cimd/client.json` on the end. The server derives `client_id` from the request,
so it self-references correctly whatever hostname the tunnel hands out.

You also need a browser, and an account that can sign in to the Studio you are
testing. Use this folder's venv as the kernel:

```bash
./.venv/bin/python -m ipykernel install --user --name cimd-interop
```


## Configure

The MCP server URL is the **only** thing this client is told about the
deployment. Everything else is discovered.

A gateway mount path carries the tenant: `/mcp/<accountId>_<projectId>/<Server>`.
Studio shows the full URL after you register an MCP server.


In [1]:
SERVER_URL = "https://gateway-dev.highflame.dev/mcp/428956146418_125bff3b-73df-44c2-ab13-1f34e6563108/notion"

# The URL for client_metadata.py, plus the document path.
DOCUMENT_URL = "https://preagricultural-breezily-jen.ngrok-free.dev/cimd/client.json"

# Where client_metadata.py listens locally. Used only to read its fetch log.
METADATA_ORIGIN = "http://127.0.0.1:8800"

# Must match CIMD_REDIRECT_URI in client_metadata.py. Both default to 8765.
CALLBACK_PORT = 8765
CALLBACK_PATH = "/callback"

# Seconds to wait for you to finish the consent page.
BROWSER_TIMEOUT = 300.0

## Imports

Standard library, the `mcp` SDK, and `httpx2` — which the SDK depends on rather
than `httpx`. `OAuthClientProvider` subclasses `httpx2.Auth`, and passing it to
an `httpx.AsyncClient` fails with `TypeError: Invalid "auth" argument`, a message
that reads like a discovery failure.


In [2]:
import base64
import json
import socket
import threading
import time
import urllib.parse
import webbrowser
from contextlib import AsyncExitStack
from dataclasses import dataclass, field
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from typing import Any

import anyio
import httpx2
from mcp import ClientSession
from mcp.client.auth import OAuthClientProvider
from mcp.client.streamable_http import streamable_http_client
from mcp.shared.auth import AuthorizationCodeResult, OAuthClientMetadata

print("ready")

ready


## Step 1 — read the document the world can see

Fetch it the way the authorization server will. Two things must hold.

**It must be reachable.** If it is not, nothing downstream works and the
authorization server's error will not say so.

**It must name itself.** `client_id` has to equal the URL it is served from.
The draft requires that self-reference (section 4) and ZeroID enforces it,
because without it anyone could point a `client_id` at someone else's document
and inherit their redirect URIs.

Note there is **no `scope`**. A client cannot know what a server will ask for
before reading that server's metadata. ZeroID builds the ephemeral client's
ceiling from this field and intersects it against whatever is requested, so a
document declaring `a b` against a server advertising `c d` fails with
`invalid_scope` — naming the identity rather than the real cause. An absent
`scope` means no client-side ceiling. Discovery decides.


In [4]:
async with httpx2.AsyncClient(timeout=20.0) as http:
    served = await http.get(DOCUMENT_URL, headers={"ngrok-skip-browser-warning": "true"})

document = served.json()
print(f"HTTP {served.status_code}")
print(json.dumps(document, indent=2))

assert served.status_code == 200, "start client_metadata.py and ngrok, then set DOCUMENT_URL"
assert document["client_id"] == DOCUMENT_URL, (
    f"the document calls itself {document['client_id']!r} but is served from "
    f"{DOCUMENT_URL!r}; the authorization server will refuse it"
)
print("\nself-reference OK")

HTTP 200
{
  "client_id": "https://preagricultural-breezily-jen.ngrok-free.dev/cimd/client.json",
  "client_name": "Highflame CIMD interop client",
  "client_uri": "https://highflame.ai",
  "redirect_uris": [
    "http://127.0.0.1:8765/callback"
  ],
  "grant_types": [
    "authorization_code",
    "refresh_token"
  ],
  "response_types": [
    "code"
  ],
  "token_endpoint_auth_method": "none"
}

self-reference OK


## Step 2 — listen for the authorization code

The redirect URI is an RFC 8252 loopback address. It is what desktop MCP
clients use, it needs no registered hostname, and it is never tunnelled.

`127.0.0.1` rather than `localhost`: section 7.3 prefers the literal address, so
a host that resolves oddly cannot change what the redirect means.

The port must match the one the document declares. The assertion below checks
that rather than letting it fail later as an opaque `invalid_request`.


In [5]:
@dataclass
class Callback:
    """What the authorization server sent back to the loopback URI."""
    code: str = ""
    state: str | None = None
    iss: str | None = None
    error: str = ""
    error_description: str = ""

    @property
    def ok(self) -> bool:
        return bool(self.code) and not self.error


received = Callback()
arrived = threading.Event()


class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def do_GET(self):
        global received
        parsed = urllib.parse.urlparse(self.path)
        if parsed.path != CALLBACK_PATH:
            self._reply(404, "Not found")
            return
        q = urllib.parse.parse_qs(parsed.query)
        first = lambda k: (q.get(k) or [""])[0]
        received = Callback(
            code=first("code"), state=first("state") or None, iss=first("iss") or None,
            error=first("error"), error_description=first("error_description"),
        )
        self._reply(200 if received.ok else 400,
                    "Authorized — close this tab" if received.ok
                    else f"{received.error}: {received.error_description}")
        arrived.set()

    def _reply(self, status, text):
        body = f"<!doctype html><meta charset=utf-8><body style=\"font:16px system-ui;margin:4rem\">{text}".encode()
        self.send_response(status)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, *a):
        return


server = ThreadingHTTPServer(("127.0.0.1", CALLBACK_PORT), Handler)
server.socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
threading.Thread(target=server.serve_forever, name="cimd-callback", daemon=True).start()

REDIRECT_URI = f"http://127.0.0.1:{CALLBACK_PORT}{CALLBACK_PATH}"
assert REDIRECT_URI in document["redirect_uris"], (
    f"the document declares {document['redirect_uris']} but this notebook listens on "
    f"{REDIRECT_URI}; set CIMD_REDIRECT_URI for client_metadata.py, or CALLBACK_PORT here"
)
print("listening on", REDIRECT_URI)


listening on http://127.0.0.1:8765/callback


## Step 3 — discovery, hop 1: the unauthenticated call

The MCP server must answer `401` and say where the rules are. The
`WWW-Authenticate` header carries `resource_metadata`, which is the only thing
the client needs to continue.

The interop OAuth matrix scores two rows here: **OAuth Protected Resource
Metadata**, and **scope parameter in WWW-Authenticate**.


In [6]:
async with httpx2.AsyncClient(timeout=20.0, follow_redirects=False) as http:
    challenge = await http.post(
        SERVER_URL, headers={"Content-Type": "application/json"},
        json={"jsonrpc": "2.0", "id": 1, "method": "initialize"},
    )

www_auth = challenge.headers.get("www-authenticate", "")
resource_metadata = ""
for part in www_auth.split(","):
    key, _, value = part.strip().partition("=")
    if key.strip().lower() == "resource_metadata":
        resource_metadata = value.strip().strip('"')

print(f"HTTP {challenge.status_code}")
print("WWW-Authenticate:", www_auth)
print("\nOPRM row:  ", "pass" if resource_metadata else "fail")
print("scope row: ", "pass" if "scope=" in www_auth else "fail — no scope parameter")
assert challenge.status_code == 401 and resource_metadata, www_auth


HTTP 401
WWW-Authenticate: Bearer realm="mcp", resource_metadata="https://gateway-dev.highflame.dev/mcp/428956146418_125bff3b-73df-44c2-ab13-1f34e6563108/notion/.well-known/oauth-protected-resource"

OPRM row:   pass
scope row:  fail — no scope parameter


## Step 4 — discovery, hop 2: protected-resource metadata

RFC 9728. This document names the authorization server, and `resource` must
round-trip against the URL you fetched. It may also declare MCP extensions; an
Enterprise-Managed Authorization entry is the EMA / ID-JAG advertisement.

Watch `scopes_supported`. The MCP scope-selection rule is: the
`WWW-Authenticate` scope, then **this field**, then the authorization server's,
then omit. Whatever stands here is what the SDK will ask for, overwriting
anything the client configured.


In [7]:
async with httpx2.AsyncClient(timeout=20.0) as http:
    oprm = (await http.get(resource_metadata)).json()

print(json.dumps(oprm, indent=2))
authorization_servers = oprm.get("authorization_servers") or []
assert authorization_servers, "no authorization server named"


{
  "resource": "https://gateway-dev.highflame.dev/mcp/428956146418_125bff3b-73df-44c2-ab13-1f34e6563108/notion",
  "authorization_servers": [
    "https://studio-dev.highflame.dev"
  ],
  "bearer_methods_supported": [
    "header"
  ],
  "scopes_supported": [
    "account:read",
    "gateway:read"
  ],
  "extensions": {
    "io.modelcontextprotocol/enterprise-managed-authorization": {}
  }
}


## Step 5 — discovery, hop 3: the authorization server's metadata

RFC 8414. **This hop decides whether CIMD happens at all.** Without
`client_id_metadata_document_supported`, a spec-compliant client falls back to
dynamic client registration and there is no CIMD row to score.

Note which URL answers. An issuer with no path component yields
`<origin>/.well-known/oauth-authorization-server`; an issuer carrying a path
yields the path-insertion form. Getting this wrong is how a client ends up
reading a gateway's own synthesized document instead of the real one.


In [ ]:
def metadata_urls(issuer: str) -> list[str]:
    """RFC 8414 section 3 discovery URLs, in the order a client tries them."""
    p = urllib.parse.urlparse(issuer.rstrip("/"))
    origin = f"{p.scheme}://{p.netloc}"
    if not p.path:
        return [f"{origin}/.well-known/oauth-authorization-server"]
    return [
        f"{origin}/.well-known/oauth-authorization-server{p.path}",
        f"{origin}/.well-known/openid-configuration{p.path}",
        f"{origin}{p.path}/.well-known/oauth-authorization-server",
    ]


issuer = authorization_servers[0]
as_metadata, used_url = {}, ""
async with httpx2.AsyncClient(timeout=20.0) as http:
    for candidate in metadata_urls(issuer):
        found = await http.get(candidate)
        if found.status_code == 200:
            as_metadata, used_url = found.json(), candidate
            break

print("answered by:", used_url, "\n")
print(json.dumps(as_metadata, indent=2))
assert as_metadata.get("client_id_metadata_document_supported"), (
    f"{used_url} does not advertise CIMD; a client would use DCR instead"
)

answered by: https://studio-dev.highflame.dev/.well-known/oauth-authorization-server 

{
  "issuer": "https://studio-dev.highflame.dev",
  "authorization_endpoint": "https://studio-dev.highflame.dev/api/cli-auth/authorize",
  "token_endpoint": "https://studio-dev.highflame.dev/api/cli-auth/token",
  "registration_endpoint": "https://studio-dev.highflame.dev/api/cli-auth/register",
  "response_types_supported": [
    "code"
  ],
  "grant_types_supported": [
    "authorization_code",
    "refresh_token"
  ],
  "token_endpoint_auth_methods_supported": [
    "none"
  ],
  "code_challenge_methods_supported": [
    "S256"
  ],
  "scopes_supported": [
    "openid",
    "profile",
    "mcp:connect",
    "mcp:tools"
  ],
  "client_id_metadata_document_supported": true
}


## Step 6 — hand the flow to the SDK

From here the `mcp` SDK does everything, and a browser opens. **Sign in and
approve the client.**

The SDK performs, in one go: discovery again from its own side, the choice of
`client_id` (the document URL), the PKCE challenge, the authorization request,
the code exchange, and the MCP handshake. It is one cell because it is one
operation; splitting it would misrepresent what happens.

`client_metadata` below mirrors the published document, built from the same
dict so the two cannot drift. When they do, the failure is a 400 about a
`redirect_uri` mismatch that points at neither file.

Watch terminal 1 for a `[doc ]` line. That is the **authorization server**
fetching your document, from its own address with a `Go-http-client` user
agent. It is the evidence the CIMD matrix row asks for.

The cell catches failure rather than raising, so the later cells can still show
you the token and the evidence.


In [9]:
@dataclass
class BrowserHandlers:
    """The two seams OAuthClientProvider hands the caller."""
    authorization_urls: list[str] = field(default_factory=list)

    async def redirect(self, authorization_url: str) -> None:
        self.authorization_urls.append(authorization_url)
        print("\nauthorization URL built by the SDK:\n", authorization_url, "\n")
        webbrowser.open(authorization_url)
        print("browser opened — sign in and approve")

    async def callback(self) -> AuthorizationCodeResult:
        deadline = time.monotonic() + BROWSER_TIMEOUT
        while time.monotonic() < deadline and not arrived.is_set():
            await anyio.sleep(0.2)
        if not arrived.is_set():
            raise TimeoutError(f"no redirect reached {REDIRECT_URI} in {BROWSER_TIMEOUT:.0f}s")
        if not received.ok:
            raise RuntimeError(f"{received.error}: {received.error_description}")
        # iss is carried through because the SDK validates the RFC 9207
        # authorization-response issuer whenever the AS advertises support.
        return AuthorizationCodeResult(code=received.code, state=received.state, iss=received.iss)

    @property
    def last_url(self) -> str | None:
        return self.authorization_urls[-1] if self.authorization_urls else None


class TokenStore:
    """The SDK's TokenStorage, in memory. A cached token would let a later run
    pass without touching the authorization server, which an interop run must
    never do."""
    def __init__(self):
        self._tokens = self._client_info = None
    async def get_tokens(self): return self._tokens
    async def set_tokens(self, t): self._tokens = t
    async def get_client_info(self): return self._client_info
    async def set_client_info(self, c): self._client_info = c


handlers, storage = BrowserHandlers(), TokenStore()
provider = OAuthClientProvider(
    server_url=SERVER_URL,
    client_metadata=OAuthClientMetadata(
        client_name=document["client_name"],
        redirect_uris=document["redirect_uris"],
        grant_types=document["grant_types"],
        response_types=document["response_types"],
        token_endpoint_auth_method=document["token_endpoint_auth_method"],
        # No scope. The SDK overwrites this from discovery anyway, and naming
        # one here would only mislead a reader into thinking it mattered.
        scope=None,
    ),
    storage=storage,
    redirect_handler=handlers.redirect,
    callback_handler=handlers.callback,
    client_metadata_url=DOCUMENT_URL,
)

stack = AsyncExitStack()
session = initialized = tools = flow_error = None
try:
    http = await stack.enter_async_context(httpx2.AsyncClient(auth=provider, timeout=60.0))
    streams = await stack.enter_async_context(streamable_http_client(SERVER_URL, http_client=http))
    session = await stack.enter_async_context(ClientSession(*streams[:2]))
    initialized = await session.initialize()
    tools = await session.list_tools()
    print("\ntools:", [t.name for t in tools.tools])
except Exception as exc:
    flow_error = exc
    print(f"\nthe MCP call did not complete: {type(exc).__name__}")
    print("keep going — the next cells show the token and say which side broke")



authorization URL built by the SDK:
 https://studio-dev.highflame.dev/api/cli-auth/authorize?response_type=code&client_id=https%3A%2F%2Fpreagricultural-breezily-jen.ngrok-free.dev%2Fcimd%2Fclient.json&redirect_uri=http%3A%2F%2F127.0.0.1%3A8765%2Fcallback&state=jRxG5WUZWRtqFSMCEDul0SoSe-q43kJmV6huVRKA_qo&code_challenge=SiAAZuVVdMdmbfoNAgTiA8lKrCHgMTIJTpTuXqmW4gw&code_challenge_method=S256&resource=https%3A%2F%2Fgateway-dev.highflame.dev%2Fmcp%2F428956146418_125bff3b-73df-44c2-ab13-1f34e6563108%2Fnotion&scope=account%3Aread+gateway%3Aread 

browser opened — sign in and approve

the MCP call did not complete: MCPError
keep going — the next cells show the token and say which side broke


## Step 7 — what the authorization server minted

Claims only; the token itself is never printed. When a resource server refuses
a token the authorization server just issued, the answer is almost always here:
`iss`, `aud`, `alg`, `kid`, or the tenant.

`aud` and `resource` should both name the exact MCP server URL. That is RFC 8707
resource binding, and it is what stops a token minted for one server working at
another.

A gateway mount path carries `<accountId>_<projectId>`, and a gateway resolves
the server entry **inside the token's tenant** — so a token for another project
makes a real server look like it does not exist.


In [ ]:
def token_claims(access_token: str) -> dict[str, Any]:
    """Decode a JWT header and payload WITHOUT verifying. A diagnostic, not a check."""
    parts = access_token.split(".")
    if len(parts) != 3:
        return {"format": "opaque, not a JWT", "length": len(access_token)}
    def seg(raw):
        return json.loads(base64.urlsafe_b64decode(raw + "=" * (-len(raw) % 4)))
    header, payload = seg(parts[0]), seg(parts[1])
    return {
        "alg": header.get("alg"), "kid": header.get("kid"), "typ": header.get("typ"),
        "iss": payload.get("iss"), "aud": payload.get("aud"), "sub": payload.get("sub"),
        "scopes": payload.get("scopes") or payload.get("scope"),
        "resource": payload.get("resource"), "exp": payload.get("exp"),
        "account_id": payload.get("account_id"), "project_id": payload.get("project_id"),
        "claim_names": sorted(payload.keys()),
    }


def tenant_from_url(url: str) -> tuple[str | None, str | None]:
    """Pull <accountId>_<projectId> out of a gateway mount path, if present."""
    for part in urllib.parse.urlparse(url).path.split("/"):
        account, sep, project = part.partition("_")
        if sep and account and project and "-" in project:
            return account, project
    return None, None


tokens = await storage.get_tokens()
if tokens is None:
    print("no token was issued — the authorization leg did not complete")
else:
    claims = token_claims(tokens.access_token)
    print(json.dumps(claims, indent=2))
    url_account, url_project = tenant_from_url(SERVER_URL)
    if url_account:
        same = claims.get("account_id") == url_account and claims.get("project_id") == url_project
        print("\ntoken tenant matches the mount:", "yes" if same else "NO")
        if not same:
            print(f"  mount: {url_account}_{url_project}")
            print(f"  token: {claims.get('account_id')}_{claims.get('project_id')}")


{
  "alg": "RS256",
  "kid": "v1",
  "typ": "JWT",
  "iss": "https://auth-dev.highflame.dev",
  "aud": [
    "https://gateway-dev.highflame.dev/mcp/428956146418_125bff3b-73df-44c2-ab13-1f34e6563108/notion"
  ],
  "sub": "user_3FX8E1SdkRNEKc7QdbfRMvWJePF",
  "scopes": [
    "account:read",
    "gateway:read"
  ],
  "resource": [
    "https://gateway-dev.highflame.dev/mcp/428956146418_125bff3b-73df-44c2-ab13-1f34e6563108/notion"
  ],
  "exp": 1791140535,
  "account_id": "428956146418",
  "project_id": "f0c76886-7c12-4da1-aaed-92819dac0ac2",
  "claim_names": [
    "account_id",
    "application_id",
    "aud",
    "client_id",
    "exp",
    "external_id",
    "grant_type",
    "iat",
    "identity_type",
    "iss",
    "jti",
    "mission_id",
    "project_id",
    "resource",
    "scopes",
    "status",
    "sub",
    "sub_type",
    "trust_level"
  ]
}

token tenant matches the mount: NO
  mount: 428956146418_125bff3b-73df-44c2-ab13-1f34e6563108
  token: 428956146418_f0c76886-7c12-4da1

: 

## Step 8 — the evidence that CIMD actually happened

`client_metadata.py` records every request it served. A fetch from the
cluster's address with a `Go-http-client` agent is the authorization server
dereferencing the `client_id`.

Your own fetch from step 1 is in this list too — it carries a `python-httpx2`
agent. Counting that as the server's would report CIMD as exercised on a run
where the server never asked for anything.

**No server fetch is not proof of failure.** ZeroID caches a resolved CIMD
client for an hour, so a second run on the same document URL legitimately shows
none. A minted token against a document-URL `client_id` is the positive proof.


In [ ]:
async with httpx2.AsyncClient(timeout=10.0) as http:
    log = (await http.get(f"{METADATA_ORIGIN}/fetches")).json()

for f in log["fetches"]:
    print(f"{f['at']}  from {f['client']:<16}  ua={f['user_agent']!r}")

ours = [f for f in log["fetches"] if "python-httpx2" in f["user_agent"]]
theirs = [f for f in log["fetches"] if f not in ours]

chosen = urllib.parse.parse_qs(
    urllib.parse.urlparse(handlers.last_url or "").query
).get("client_id", [None])[0]

print("\nclient_id on the wire:", chosen)
print("matches the document: ", chosen == DOCUMENT_URL)
print("fetches by the server:", len(theirs))
print("\nCIMD row:", "pass" if chosen == DOCUMENT_URL and tokens is not None else "fail")


## Step 9 — if the MCP call failed, say which side broke

The SDK collapses any non-2xx whose body is not a JSON-RPC error into
`INTERNAL_ERROR: Server returned an error response`, discarding the status and
the body. Those two are the whole diagnosis, so fetch them once, directly.

How to read a 401 from a Highflame gateway:

| Body | Meaning |
|---|---|
| `jwt_required` | the server is registered; your credential was not accepted |
| `unauthorized`, *Use OAuth 2.0 or API key* | the gateway could not resolve the server **for this caller** |

The second is not a token rejection. A registered server answers `jwt_required`
for any credential, good or junk.


In [ ]:
if flow_error is None:
    print("the MCP call succeeded; nothing to diagnose")
elif tokens is None:
    print("no token was minted, so the failure is in the authorization leg")
else:
    async with httpx2.AsyncClient(timeout=20.0) as http:
        reply = await http.post(
            SERVER_URL,
            headers={
                "Authorization": f"Bearer {tokens.access_token}",
                "Content-Type": "application/json",
                "Accept": "application/json, text/event-stream",
            },
            json={"jsonrpc": "2.0", "id": 1, "method": "initialize",
                  "params": {"protocolVersion": "2026-07-28", "capabilities": {},
                             "clientInfo": {"name": "cimd-notebook", "version": "1.0"}}},
        )
        body = (await reply.aread()).decode(errors='replace')
    print(f"HTTP {reply.status_code}")
    print("WWW-Authenticate:", reply.headers.get("www-authenticate", ""))
    print("body:", body[:800])


## Step 10 — call a tool

Only meaningful when step 6 succeeded. `session` is still open, held by the
`AsyncExitStack` from that cell.


In [ ]:
if session is not None and tools is not None and tools.tools:
    name = tools.tools[0].name
    result = await session.call_tool(name, {})
    print(f"called {name}:", type(result).__name__)
else:
    print("no live session, or no tools listed — skipping")


## Clean up

Run this even if a step failed. It closes the MCP session and releases the
loopback port, so the next run can bind it.

`client_metadata.py` and ngrok keep running in their own terminals. Leave them
up for the next run; the document URL stays valid.


In [ ]:
await stack.aclose()
server.shutdown()
server.server_close()
print("session closed, callback port released")
